In [24]:
import pandas as pd
import numpy as np
import kagglehub
import random
import torch
import torch.nn as nn
import re
import os
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

from transformers import AutoTokenizer, AutoModel, AutoConfig
from torch.utils.data import Dataset, DataLoader
from transformers.utils import logging
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor


In [2]:
class cfg:
    competition = 'learning-agency-lab-automated-essay-scoring-2'
    checkpoint = '/kaggle/input/notebooks/michelliez/torch-deberta-v3-small/deberta_v3_small'    
    tokenizer = None
    max_length = 512
    batch_size = 16
    epochs = 10
    learning_rate = 5e-5
    lgbm_lr = 0.03
    weight_decay = 0.01
    task = 'regression'

In [3]:
#Quadratic Weighted Kappa score
def get_score(y_true, y_pred):
    return cohen_kappa_score(
        y_true,
        y_pred,
        weights="quadratic"
    )

In [4]:
#Sets random to 42
def seed_everything(seed: int=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    
seed_everything(42)

In [5]:
path = kagglehub.competition_download(cfg.competition)

In [6]:
train_df = pd.read_csv(f'{path}/train.csv')

In [7]:
train_df['word_count'] = train_df['full_text'].str.split(' ').str.len()

train_df['num_sentences'] = train_df['full_text'].apply(lambda x: len([s for s in re.split(r'[.!?]+', str(x)) if s.strip()]))

train_df['sentence_length'] = train_df[['num_sentences', 'word_count']].apply(lambda row: row['word_count']/row['num_sentences'], axis=1)

train_df['num_para'] = train_df['full_text'].apply(lambda x: len([p for p in str(x).split('\n') if p.strip()]))

train_df['para_length'] = train_df [['num_para', 'word_count']].apply(lambda row: row['word_count']/row['num_para'], axis=1)

train_df['long_essay'] = (train_df['word_count']>2500).astype(int)

train_df["unique_word_ratio"] = train_df["full_text"].apply(lambda x: len(set(x.lower().split())) / max(len(x.split()), 1))

In [8]:
feature_cols = ['word_count', 'num_sentences', 'sentence_length', 'num_para', 'para_length', 'long_essay', 'unique_word_ratio']

In [9]:
#Label y starting from 0 so classifier works
if train_df['score'].min() == 1:
    train_df['score'] = train_df['score'] - 1

In [10]:
#Cross Validation
train_df['fold'] = -1
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, valid_idx) in enumerate(skf.split(train_df, train_df['score'])):
    train_df.loc[valid_idx, 'fold'] = fold

In [11]:
tokenizer = AutoTokenizer.from_pretrained(cfg.checkpoint)

In [12]:
class EssayDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=cfg.max_length):
        self.X = data['full_text']
        if 'score' in data.columns:
            self.y = data['score']
        else:
            self.y = None
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        tokens = self.tokenizer(
            text = self.X.iloc[idx],
            max_length=self.max_length,
            truncation = True,
            padding = 'max_length',
            return_tensors = 'pt'
        )
        item = {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
        }
        if self.y is not None:
            item['labels'] = torch.tensor(self.y.iloc[idx], dtype=torch.float)
        return item


In [13]:
#MeanPooling code taken from online resources
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()

    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_embeddings/sum_mask
        return mean_embeddings

In [14]:
class EssayModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained(cfg.checkpoint)
        self.mp = MeanPooling()
        self.classifier = nn.Sequential(
            nn.Linear(self.model.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids = input_ids, attention_mask = attention_mask)
        mp_output = self.mp(output.last_hidden_state, attention_mask)
        logits = self.classifier(mp_output)
        return logits


In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
#Get out of fold predictions on validation sets based on DeBERTa small model
oof_pred = np.zeros(len(train_df))
for fold in range(5):
    valid_idx = train_df[train_df['fold']==fold].index
    valid_df = train_df.loc[valid_idx].reset_index(drop=True)

    valid_dataset = EssayDataset(valid_df, tokenizer)
    valid_loader = DataLoader(valid_dataset, batch_size=cfg.batch_size, shuffle=False)

    logging.set_verbosity_error()
    deberta_model = EssayModel().to(device)
    deberta_model.load_state_dict(
        torch.load(f'/kaggle/input/notebooks/michelliez/deberta-mlp/best_deberta_model_fold_{fold}.pth', map_location=device)
    )
    deberta_model.eval()

    preds = []
    with torch.no_grad():
            for batch in valid_loader:
                logits = deberta_model(batch['input_ids'].to(device), batch['attention_mask'].to(device)).squeeze(-1)
                preds.extend(logits.cpu().numpy())
                
    oof_pred[valid_idx] = preds
    del deberta_model

train_df['mlp_oof'] = oof_pred
np.save('mlp_oof.npy', oof_pred)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [45]:

features = ['mlp_oof', 'word_count', 'num_sentences', 'sentence_length', 'num_para', 'para_length', 'long_essay', 'unique_word_ratio']
y = train_df['score']

lgb_oof = np.zeros(len(train_df))
lgb_models = []

xgb_oof = np.zeros(len(train_df))
xgb_models = []

for fold in range (5):
    print(f'-------Fold {fold}-------')
    train_idx = train_df[train_df['fold']!=fold].index
    valid_idx = train_df[train_df['fold']==fold].index

    X_train = train_df.loc[train_idx, features]
    y_train = train_df.loc[train_idx, 'score']
    X_valid = train_df.loc[valid_idx, features]
    y_valid = train_df.loc[valid_idx, 'score']
  
    lgb_model = LGBMRegressor(
        objective=cfg.task,
        learning_rate=0.02,
        n_estimators=600,
        num_leaves=32,
        max_depth=4,
        random_state=42,
        verbosity=-1
    )

    lgb_model.fit(X_train, y_train)
    lgb_valid_preds = lgb_model.predict(X_valid)
    lgb_oof[valid_idx] = lgb_valid_preds
    lgb_fold_qwk = get_score(y_valid, np.round(lgb_valid_preds).clip(0,5).astype(int))
    print('Fold LGB QWK:', lgb_fold_qwk)
    lgb_models.append(lgb_model)

    xgb_model = XGBRegressor(
        objective = 'reg:squarederror',
        learning_rate= 0.02,
        n_estimators = 800,
        max_depth = 3,
        min_child_weight=5,
        subsample = 0.9,
        reg_lambda = 0,
        reg_alpha = 0,
        random_state=42,
    )
    xgb_model.fit(X_train, y_train)
    xgb_valid_preds = xgb_model.predict(X_valid)
    xgb_oof[valid_idx] = xgb_valid_preds
    xgb_fold_qwk = get_score(y_valid, np.round(xgb_valid_preds).clip(0,5).astype(int))
    print('Fold XGB QWK:', xgb_fold_qwk)
    xgb_models.append(xgb_model)

overall_lgb_qwk = get_score(y, np.round(lgb_oof).clip(0,5).astype(int))
print('Overall LGB OOF QWK:', overall_lgb_qwk)

overall_xgb_qwk = get_score(y, np.round(xgb_oof).clip(0,5).astype(int))
print('Overall XGB OOF QWK:', overall_xgb_qwk)

for i, model in enumerate(lgb_models):
    joblib.dump(model, f'lgb_model_fold_{i}.pkl')
np.save('lgb_oof.npy', lgb_oof)

for i, model in enumerate(xgb_models):
    joblib.dump(model, f'xgb_model_fold_{i}.pkl')
np.save('xgb_oof.npy', xgb_oof)

ensemble_oof = 0.5*lgb_oof + 0.5*xgb_oof

-------Fold 0-------
Fold LGB QWK: 0.797735042797416
Fold XGB QWK: 0.7996020329118714
-------Fold 1-------
Fold LGB QWK: 0.8024699147398642
Fold XGB QWK: 0.8051564932301121
-------Fold 2-------
Fold LGB QWK: 0.7781435419820517
Fold XGB QWK: 0.7813897988070587
-------Fold 3-------
Fold LGB QWK: 0.796067432865365
Fold XGB QWK: 0.7970944265322403
-------Fold 4-------
Fold LGB QWK: 0.793533482053202
Fold XGB QWK: 0.7945102635062047
Overall LGB OOF QWK: 0.7935886195995914
Overall XGB OOF QWK: 0.7955537597536289


In [18]:
def find_thresholds(true, pred, steps=50):
    xs = [[], [], [], [], []]
    ys = [[], [], [], [], []]

    threshold = [0.5, 1.5, 2.5, 3.5, 4.5]

    pred2 = pd.cut(
        pred,
        [-np.inf] + threshold + [np.inf],
        labels=[0, 1, 2, 3, 4, 5]
    ).astype("int32")

    best = cohen_kappa_score(true, pred2, weights="quadratic")

    for k in range(5):
        for sign in [1, -1]:
            v = threshold[k]
            threshold2 = threshold.copy()
            stop = 0

            while stop < steps:
                v += sign * 0.001
                threshold2[k] = v

                pred2 = pd.cut(
                    pred,
                    [-np.inf] + threshold2 + [np.inf],
                    labels=[0, 1, 2, 3, 4, 5]
                ).astype("int32")

                metric = cohen_kappa_score(true, pred2, weights="quadratic")

                xs[k].append(v)
                ys[k].append(metric)

                if metric <= best:
                    stop += 1
                else:
                    stop = 0
                    best = metric
                    threshold = threshold2.copy()

    threshold = [np.round(t, 3) for t in threshold]
    return best, threshold, xs, ys

In [46]:
best, thresholds, xs, ys = find_thresholds(
    train_df['score'].values,
    ensemble_oof,
    steps=500
)

print("Best thresholds are:", thresholds)
print("=> achieve Overall CV QWK score =", best)

Best thresholds are: [np.float64(0.805), np.float64(1.561), np.float64(2.358), np.float64(3.184), np.float64(4.188)]
=> achieve Overall CV QWK score = 0.8109983751578779
